# Course 4: Mathematics and AI
## Session 11: Autoformalization

**Goal:** practise catching formal statements that do not say what the informal statement means, then autoformalize ten textbook statements into Lean.

**How to use this notebook:** run the cells from top to bottom. Cells marked `TODO` are for you to complete. The lab guide, quiz and assignment for this session are in the course document.

**Testing note:** every Python cell without a NOT RUN label was run once on a CPU (November 2026). Cells labelled NOT RUN need a library, a GPU or a Lean installation. Lean code in this course was written carefully but was NOT compiled here: check it in Lean before you teach it.

### 1. The problem
Autoformalization turns a statement in ordinary mathematical language into a formal one. A proof assistant can check a proof. It cannot check that the formal statement matches what the author meant. A statement that is subtly wrong is often easier to prove, so a system that is rewarded for proofs will drift towards wrong statements unless someone checks faithfulness.

### 2. A faithfulness test you can run: search for counterexamples
Below, each informal statement comes with two candidate formalizations written as Python predicates over small numbers. One is faithful. The other has a typical autoformalization error. Testing on small cases exposes many of them.

In [ ]:
import itertools
N = range(0, 30)
is_prime = lambda n: n >= 2 and all(n % d for d in range(2, int(n ** 0.5) + 1))
cases = [
 {"informal": "Every even number greater than 2 is the sum of two primes (checked up to 28).",
  "A": lambda: all(any(is_prime(p) and is_prime(n - p) for p in N) for n in N if n % 2 == 0 and n > 2),
  "B": lambda: all(any(is_prime(p) and is_prime(n - p) for p in N) for n in N if n % 2 == 0),            # forgot 'greater than 2'
  "faithful": "A"},
 {"informal": "For all natural numbers a and b, if a divides b and b divides a, then a = b.",
  "A": lambda: all(a == b for a in N for b in N if a and b and b % a == 0 and a % b == 0),
  "B": lambda: all(a == b for a in N for b in N if a and b and b % a == 0),                              # dropped a hypothesis
  "faithful": "A"},
 {"informal": "There is a natural number that is larger than every natural number below 30.",
  "A": lambda: all(any(m > n for m in range(40)) for n in N),                                           # swapped the quantifiers
  "B": lambda: any(all(m > n for n in N) for m in range(40)),
  "faithful": "B"},
 {"informal": "The square of an odd number is odd.",
  "A": lambda: all((n * n) % 2 == 1 for n in N if n % 2 == 1),
  "B": lambda: all(n % 2 == 1 for n in N if (n * n) % 2 == 1),                                          # proved the converse instead
  "faithful": "A"},
 {"informal": "For natural numbers, a - b + b = a whenever b <= a.",
  "A": lambda: all(max(a - b, 0) + b == a for a in N for b in N if b <= a),
  "B": lambda: all(max(a - b, 0) + b == a for a in N for b in N),                                        # natural-number subtraction truncates at 0
  "faithful": "A"},
]
for i, c in enumerate(cases, 1):
    print(f"{i}. {c['informal']}\n   candidate A holds on small cases: {c['A']()} | candidate B holds: {c['B']()} | faithful: {c['faithful']}")

What the output shows:
- Cases 1, 2 and 5: the wrong candidate is false on small numbers, so testing catches it.
- Case 3: **both candidates are true**, yet they say different things. 'For every n there is a larger m' is not 'there is one m larger than all n'. Here the second happens to be true only because the range is finite. Testing cannot separate them.
- Case 4: both are true, and the wrong one is the converse. A proof of B is a proof of a different theorem.

So counterexample search is a cheap first filter and not a guarantee.

In [ ]:
caught = sum(c["A"]() != c["B"]() for c in cases)
print(f"wrong formalizations caught by testing on small cases: {caught} of {len(cases)}")
assert caught == 3

### 3. The typical errors, in Lean terms
1. **Natural-number subtraction.** In Lean, `(3 : ℕ) - 5 = 0`. A statement with `-` over `ℕ` often needs `ℤ` or a hypothesis `b ≤ a`.
2. **Division.** `(7 : ℕ) / 2 = 3`, and `x / 0 = 0` by convention. 'Average' or 'ratio' needs `ℚ` or `ℝ`.
3. **Missing hypotheses.** 'Let p be an odd prime' becomes `(p : ℕ)` with nothing else.
4. **Quantifier order and scope.** `∀ ε, ∃ δ` against `∃ δ, ∀ ε`.
5. **Vacuous truth.** Contradictory hypotheses make any conclusion provable. If a hard problem is proved by `simp`, suspect the statement.
6. **Wrong type.** `x^2 = 2` has no solution in `ℚ` and two in `ℝ`.

### 4. A review checklist as code
Keep one record per statement. A formalization is accepted only when every check is done and recorded.

In [ ]:
import json
def new_record(i, informal):
    return {"id": i, "informal": informal, "lean": "", "compiles": None, "back_translation": "", "faithful": None,
            "checks": {"types_right": None, "all_hypotheses_present": None, "quantifiers_right": None, "not_vacuous": None, "small_cases_tested": None},
            "notes": ""}

STATEMENTS = [
 "The sum of two even integers is even.",
 "For every natural number n, n squared is greater than or equal to n.",
 "If a divides b and b divides c, then a divides c (natural numbers).",
 "There are infinitely many prime numbers.",
 "The square root of 2 is irrational.",
 "For real x, x squared is non-negative.",
 "A function with a left inverse is injective.",
 "The composition of two injective functions is injective.",
 "In a group, the inverse of the inverse of a is a.",
 "For sets A and B, the intersection A ∩ B is a subset of A.",
]
records = [new_record(i + 1, s) for i, s in enumerate(STATEMENTS)]

def accepted(r): return bool(r["lean"]) and r["compiles"] is True and r["faithful"] is True and all(v is True for v in r["checks"].values())
def summary(records):
    return {"total": len(records), "compiles": sum(r["compiles"] is True for r in records), "accepted": sum(accepted(r) for r in records)}

# One worked example. The Lean line was written by hand and NOT compiled here: check it yourself.
records[0].update(lean="theorem even_add_even (a b : ℤ) (ha : Even a) (hb : Even b) : Even (a + b) := by sorry",
                  back_translation="For integers a and b, if a is even and b is even then a + b is even.", notes="statement only; proof left as sorry")
print(json.dumps(records[0], ensure_ascii=False, indent=1)); print(summary(records))
assert summary(records)["accepted"] == 0             # nothing counts as accepted until a person has filled in the checks

### 5. Autoformalize the ten statements (NOT RUN here)

In [ ]:
# NOT RUN: needs a language model (any capable chat model) and a Lean 4 + Mathlib installation or the web editor.
PROMPT = """Translate the statement into a Lean 4 theorem using Mathlib. Give only the theorem statement ending in `:= by sorry`.
Use ℤ, ℚ or ℝ instead of ℕ whenever subtraction or division appears, and include every hypothesis.
Statement: {informal}"""
# for r in records:
#     r["lean"] = llm(PROMPT.format(informal=r["informal"]))
#     r["compiles"] = lean_check(r["lean"])                       # statement must type-check with `sorry` as the proof
#     r["back_translation"] = llm("State in plain English exactly what this Lean theorem says:\n" + r["lean"])
# Then, by hand, for every record: compare the back-translation with the informal statement and fill in the five checks.
# Report: how many compiled, how many were faithful, and the error type (from section 3) for each failure.

### 6. Back-translation is a filter, not a judge
Asking a model to translate the Lean statement back into English and comparing the two is the usual automatic check. It misses errors that survive the round trip, such as the natural-number subtraction in case 5, where the English reads the same. Use it to sort candidates and keep a person in the loop for acceptance.

### TODO
1. Complete the ten records. For each rejected formalization, name the error type.
2. Write a sixth case for section 2 where both candidates pass small-case testing but only one is faithful.
3. For three accepted statements, replace `sorry` with a real proof (try `simp`, `omega`, `exact?` first).